Język: [English](examples.en.ipynb) | **Polski**

# Wykład 11 — eksperyment: od rozkładu prawdopodobieństwa do generowania tekstu

Materiał towarzyszący [`lecture.md`](lecture.md). Pięć eksperymentów: (A)
tokenizacja; (B) next-token prediction, logity, ręczne generowanie vs
`model.generate()`; (C) strategie dekodowania; (D) chat templates; (E) KV
cache i koszt.

**Uwaga: ten notebook pobiera wagi modelu (ok. 257MB) przy pierwszym
uruchomieniu** — wymaga internetu. Kolejne uruchomienia używają lokalnego
cache Hugging Face.


In [1]:
import time

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Uzywane urzadzenie: {device}")

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
REVISION = "12fd25f77366fa6b3b4b768ec3050bf629380bac"  # przypiete, nie "najnowsza wersja"


/Users/michalmaj/Documents/python/deep-learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Uzywane urzadzenie: cpu


## A. Tokenizacja

Ten sam tokenizer, pięć bardzo różnych przykładów tekstu.


In [2]:
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
print(f"tokenizer load: {time.time() - t0:.2f}s")


tokenizer load: 0.52s


In [3]:
examples = {
    "angielskie zdanie": "The quick brown fox jumps over the lazy dog.",
    "polskie zdanie": "Szybki brązowy lis przeskakuje nad leniwym psem.",
    "liczby/daty": "The invoice total is 1234.56 USD, due on 2026-03-15.",
    "znaki specjalne": "Email me at test@example.com -- see §4.2 for *details* (urgent!!).",
    "rzadkie slowa": (
        "The spectrophotometric quokka antidisestablishmentarianism "
        "supercalifragilisticexpialidocious."
    ),
}

print(f"{'przyklad':20s} {'znaki':>6s} {'slowa':>6s} {'tokeny':>7s} {'slowa/token':>12s}")
for name, text in examples.items():
    ids = tokenizer(text)["input_ids"]
    n_chars, n_words = len(text), len(text.split())
    print(f"{name:20s} {n_chars:>6d} {n_words:>6d} {len(ids):>7d} {n_words/len(ids):>12.2f}")


przyklad              znaki  slowa  tokeny  slowa/token
angielskie zdanie        44      9      10         0.90
polskie zdanie           48      7      25         0.28
liczby/daty              52      9      28         0.32
znaki specjalne          66     10      23         0.43
rzadkie slowa            94      5      25         0.20


In [4]:
# Zajrzyjmy do konkretnych tokenow dla dwoch najbardziej kontrastujacych przykladow
for name in ["angielskie zdanie", "polskie zdanie"]:
    text = examples[name]
    ids = tokenizer(text)["input_ids"]
    tokens = tokenizer.convert_ids_to_tokens(ids)
    print(f"--- {name} ---")
    print(f"tekst: {text!r}")
    print(f"tokeny ({len(tokens)}): {tokens}")
    print(f"id: {ids}")
    print()


--- angielskie zdanie ---
tekst: 'The quick brown fox jumps over the lazy dog.'
tokeny (10): ['The', 'Ġquick', 'Ġbrown', 'Ġfox', 'Ġjumps', 'Ġover', 'Ġthe', 'Ġlazy', 'Ġdog', '.']
id: [504, 2365, 6354, 16438, 27003, 690, 260, 23790, 2767, 30]

--- polskie zdanie ---
tekst: 'Szybki brązowy lis przeskakuje nad leniwym psem.'
tokeny (25): ['S', 'zy', 'b', 'ki', 'Ġbr', 'Ä', 'ħ', 'z', 'ow', 'y', 'Ġl', 'is', 'Ġpr', 'zes', 'k', 'aku', 'je', 'Ġn', 'ad', 'Ġlen', 'iw', 'ym', 'Ġp', 'sem', '.']
id: [67, 5346, 82, 8139, 1156, 143, 223, 106, 323, 105, 303, 271, 694, 16244, 91, 28277, 16864, 304, 344, 3858, 17493, 1905, 273, 8188, 30]



**Obserwacja.** Angielskie zdanie: ok. 1 token na słowo. Polskie zdanie: ok.
**3-4 tokeny na słowo** — ten sam tokenizer, dramatycznie różna granulacja,
bo był trenowany głównie na angielskim tekście/kodzie. **Nie można zakładać
stałego stosunku słowa:tokeny, zwłaszcza między językami.**


In [5]:
# liczby i znaki specjalne - szczegoly
for name in ["liczby/daty", "znaki specjalne", "rzadkie slowa"]:
    text = examples[name]
    tokens = tokenizer.convert_ids_to_tokens(tokenizer(text)["input_ids"])
    print(f"--- {name} ---")
    print(f"tekst: {text!r}")
    print(f"tokeny: {tokens}")
    print()

print(
    "Obserwacje: cyfry sa czesto dzielone POJEDYNCZO (kazda cyfra to osobny token); "
    "znaki specjalne poza ASCII moga dawac dziwnie wygladajace tokeny (bajtowe "
    "kodowanie BPE); rzadkie/dlugie slowa sa rozbijane na sensowne podslowa, NIE "
    "zamieniane na jeden token '<UNK>' - to jest wlasnie zaleta tokenizacji subword."
)


--- liczby/daty ---
tekst: 'The invoice total is 1234.56 USD, due on 2026-03-15.'
tokeny: ['The', 'Ġinvoice', 'Ġtotal', 'Ġis', 'Ġ', '1', '2', '3', '4', '.', '5', '6', 'ĠUSD', ',', 'Ġdue', 'Ġon', 'Ġ', '2', '0', '2', '6', '-', '0', '3', '-', '1', '5', '.']

--- znaki specjalne ---
tekst: 'Email me at test@example.com -- see §4.2 for *details* (urgent!!).'
tokeny: ['Email', 'Ġme', 'Ġat', 'Ġtest', '@', 'example', '.', 'com', 'Ġ--', 'Ġsee', 'ĠÂ§', '4', '.', '2', 'Ġfor', 'Ġ*', 'details', '*', 'Ġ(', 'urg', 'ent', '!!', ').']

--- rzadkie slowa ---
tekst: 'The spectrophotometric quokka antidisestablishmentarianism supercalifragilisticexpialidocious.'
tokeny: ['The', 'Ġspect', 'roph', 'ot', 'ometric', 'Ġqu', 'ok', 'ka', 'Ġant', 'idis', 'establish', 'ment', 'arianism', 'Ġsuper', 'cal', 'if', 'rag', 'il', 'istice', 'x', 'p', 'ial', 'id', 'ocious', '.']

Obserwacje: cyfry sa czesto dzielone POJEDYNCZO (kazda cyfra to osobny token); znaki specjalne poza ASCII moga dawac dziwnie wygladajace tokeny (

## B. Next-token prediction

Ładujemy model. Dla krótkiego promptu pobieramy logity ostatniej pozycji,
liczymy rozkład, patrzymy na wpływ temperatury, generujemy ręcznie krok po
kroku i porównujemy z `model.generate()`.


In [6]:
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, dtype=torch.float32)
model.eval()
print(f"model load: {time.time() - t0:.2f}s")
print(f"liczba parametrow: {sum(p.numel() for p in model.parameters()):,}")
print(model.config)


W1017 11:32:16.059000 94986 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/272 [00:00<00:50,  5.39it/s]

Loading weights:  87%|████████▋ | 237/272 [00:00<00:00, 1026.94it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 894.09it/s] 

model load: 0.53s
liczba parametrow: 134,515,008
LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "float32",
  "eos_token_id": 2,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 576,
  "initializer_range": 0.041666666666666664,
  "intermediate_size": 1536,
  "is_llama_config": true,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 9,
  "num_hidden_layers": 30,
  "num_key_value_heads": 3,
  "pad_token_id": 2,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_interleaved": false,
  "rope_parameters": {
    "rope_theta": 100000,
    "rope_type": "default"
  },
  "tie_word_embeddings": true,
  "transformers.js_config": {
    "kv_cache_dtype": {
      "fp16": "float16",
      "q4f16": "float16"
    }
  },
  "transformers_version": "5.20.1",
  "use_cache": true,
  "vocab_size": 49152
}



In [7]:
messages = [{"role": "user", "content": "The capital of France is"}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True
)
prompt_len = inputs["input_ids"].shape[1]
print(f"prompt (po chat template): {prompt_len} tokenow")
print(tokenizer.decode(inputs["input_ids"][0]))


prompt (po chat template): 35 tokenow
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
The capital of France is<|im_end|>
<|im_start|>assistant



In [8]:
with torch.no_grad():
    out = model(**inputs)
next_token_logits = out.logits[0, -1, :]
print(f"logity ksztalt: {next_token_logits.shape}")

probs = F.softmax(next_token_logits, dim=-1)
topk_probs, topk_ids = probs.topk(5)
print("\n5 najbardziej prawdopodobnych nastepnych tokenow:")
for p, idx in zip(topk_probs.tolist(), topk_ids.tolist()):
    print(f"  id={idx:>6d}  token={tokenizer.decode([idx])!r:12s}  p={p:.4f}")


logity ksztalt: torch.Size([49152])

5 najbardziej prawdopodobnych nastepnych tokenow:
  id=   504  token='The'         p=0.3113
  id=    50  token='B'           p=0.0674
  id=    18  token='"'           p=0.0616
  id= 30904  token='Bon'         p=0.0353
  id= 38634  token='Paris'       p=0.0344


In [9]:
print("Wplyw temperatury na ten sam rozklad logitow:")
for tau in [0.5, 1.0, 2.0]:
    p_tau = F.softmax(next_token_logits / tau, dim=-1)
    top3 = p_tau.topk(3)
    pairs = zip(top3.values.tolist(), top3.indices.tolist())
    tokens_str = [f"{tokenizer.decode([i])!r}:{v:.3f}" for v, i in pairs]
    print(f"  tau={tau}: top-3 {tokens_str}")

print(
    "\nMniejsze tau -> wieksza koncentracja na juz-najbardziej-prawdopodobnym tokenie; "
    "wieksze tau -> rozklad placszy. To TA SAMA logity, inna transformacja."
)


Wplyw temperatury na ten sam rozklad logitow:
  tau=0.5: top-3 ["'The':0.866", "'B':0.041", '\'"\':0.034']
  tau=1.0: top-3 ["'The':0.311", "'B':0.067", '\'"\':0.062']
  tau=2.0: top-3 ["'The':0.026", "'B':0.012", '\'"\':0.012']

Mniejsze tau -> wieksza koncentracja na juz-najbardziej-prawdopodobnym tokenie; wieksze tau -> rozklad placszy. To TA SAMA logity, inna transformacja.


In [10]:
# reczne generowanie krok po kroku (greedy)
n_steps = 10
manual_ids = inputs["input_ids"].clone()
manual_tokens = []
with torch.no_grad():
    for step in range(n_steps):
        step_out = model(manual_ids)
        next_id = step_out.logits[0, -1, :].argmax().item()
        manual_tokens.append(next_id)
        manual_ids = torch.cat([manual_ids, torch.tensor([[next_id]])], dim=1)
        if next_id == tokenizer.eos_token_id:
            print(f"EOS wygenerowany w kroku {step} - zatrzymujemy sie")
            break

manual_text = tokenizer.decode(manual_tokens, skip_special_tokens=True)
print(f"reczne tokeny: {manual_tokens}")
print(f"reczny tekst: {manual_text!r}")


EOS wygenerowany w kroku 7 - zatrzymujemy sie
reczne tokeny: [504, 3575, 282, 4649, 314, 7042, 30, 2]
reczny tekst: 'The capital of France is Paris.'


In [11]:
# model.generate() z odpowiadajacymi ustawieniami - TEN SAM prompt (bez modyfikacji formatowania)
with torch.no_grad():
    gen_out = model.generate(
        input_ids=inputs["input_ids"], max_new_tokens=n_steps, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
generate_tokens = gen_out[0, prompt_len:].tolist()
generate_text = tokenizer.decode(generate_tokens, skip_special_tokens=True)
print(f"generate() tokeny: {generate_tokens}")
print(f"generate() tekst: {generate_text!r}")

match = manual_tokens == generate_tokens[: len(manual_tokens)]
print(f"\nreczne tokeny == generate() tokeny: {match}")
assert match
print(
    "Zgodnosc potwierdzona: 'generate()' z do_sample=False to (matematycznie) "
    "dokladnie ta sama petla, co reczne wywolania model() + argmax + konkatenacja."
)


generate() tokeny: [504, 3575, 282, 4649, 314, 7042, 30, 2]
generate() tekst: 'The capital of France is Paris.'

reczne tokeny == generate() tokeny: True
Zgodnosc potwierdzona: 'generate()' z do_sample=False to (matematycznie) dokladnie ta sama petla, co reczne wywolania model() + argmax + konkatenacja.


### `pipeline()` jako wygodna abstrakcja (krótko)

`transformers.pipeline` chowa tokenizację, `generate()` i dekodowanie za
jednym wywołaniem — wygodne w praktyce, ale **nie** używamy go jako głównego
narzędzia w tym notebooku, bo chowa dokładnie te etapy, które chcemy widzieć.


In [12]:
from transformers import pipeline

# pipeline() auto-detects an accelerator (np. MPS na Mac) domyslnie - jawne
# device=device trzyma je w zgodzie z reszta notebooka (CPU, zobacz sekcja 0)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=device)
pipe_out = pipe(messages, max_new_tokens=10, do_sample=False)
pipe_text = pipe_out[0]["generated_text"][-1]["content"]
print(f"pipeline() tekst: {pipe_text!r}")
print(f"reczne/generate() tekst (z wyzej): {manual_text!r}")
print(
    "Ten sam wynik, trzy linie kodu - ale 'co dokladnie sie dzieje' jest schowane. "
    "W tym module uzywamy generate() jako glownego narzedzia wlasnie dlatego, ze "
    "etapy (tokenizacja, logity, wybor tokenu, dekodowanie) musza byc widoczne."
)


pipeline() tekst: 'The capital of France is Paris.'
reczne/generate() tekst (z wyzej): 'The capital of France is Paris.'
Ten sam wynik, trzy linie kodu - ale 'co dokladnie sie dzieje' jest schowane. W tym module uzywamy generate() jako glownego narzedzia wlasnie dlatego, ze etapy (tokenizacja, logity, wybor tokenu, dekodowanie) musza byc widoczne.


## C. Strategie dekodowania

Porównujemy greedy, sampling (różne temperatury), top-k, top-p na tych samych
promptach, z tym samym budżetem długości i kontrolowanym stanem losowości.


In [13]:
decoding_prompt = [{"role": "user", "content":
    "Tell me a short, imaginative story about a robot who discovers music."}]
dec_inputs = tokenizer.apply_chat_template(
    decoding_prompt, tokenize=True, add_generation_prompt=True,
    return_tensors="pt", return_dict=True,
)
dec_prompt_len = dec_inputs["input_ids"].shape[1]

configs = [
    ("greedy", dict(do_sample=False)),
    ("sampling, tau=0.7", dict(do_sample=True, temperature=0.7)),
    ("sampling, tau=1.5", dict(do_sample=True, temperature=1.5)),
    ("top-k=10", dict(do_sample=True, temperature=1.0, top_k=10)),
    ("top-p=0.9", dict(do_sample=True, temperature=1.0, top_k=0, top_p=0.9)),
]

print("=== otwarty, kreatywny prompt ===")
for name, kwargs in configs:
    torch.manual_seed(0)
    with torch.no_grad():
        out = model.generate(**dec_inputs, max_new_tokens=30,
                              pad_token_id=tokenizer.pad_token_id, **kwargs)
    text = tokenizer.decode(out[0, dec_prompt_len:], skip_special_tokens=True)
    print(f"[{name}]\n  {text!r}\n")


=== otwarty, kreatywny prompt ===


[greedy]
  'In the heart of a bustling city, a young robot named Kael stumbled upon a hidden underground music scene. The sounds of a thousand instruments filled the'



[sampling, tau=0.7]
  "The music of the gods. So popular, so enchanted, it's like a whisper in the ear of a thousand gods. One day, a"



[sampling, tau=1.5]
  'The life of a robot seems simple but never truly ends, and not all robots thrive without exploring their true passions after they have begun their digital escapades'



[top-k=10]
  "The music of the robot. It's the promise of limitless creative possibilities, the symphony of human imagination, and the whispered promises of a new era in"



[top-p=0.9]
  'The music of the gods. So popular, so enchanted. A young princess named Lyra (heavily piped her as Lily, Lily'



**Zadanie diagnostyczne dla studenta (markdown, nie kod)**: zauważ, że
`top-k=0` w konfiguracji `top-p=0.9` jest celowe — `top_k` domyślnie **nie**
jest wyłączony, więc gdybyśmy go nie wyzerowali, obie filtracje działałyby
**równocześnie**, i wynik nie izolowałby efektu samego top-p. To jest
dokładnie "brak przypadkowego wykorzystania ustawień ignorowanych w danym
trybie" z sekcji Eksperymentu C w `lecture.md`.


In [14]:
print("=== skoncentrowany, faktograficzny prompt (dla porownania) ===")
factual_prompt = [{"role": "user", "content": "What is the capital of France?"}]
fact_inputs = tokenizer.apply_chat_template(
    factual_prompt, tokenize=True, add_generation_prompt=True,
    return_tensors="pt", return_dict=True,
)
fact_prompt_len = fact_inputs["input_ids"].shape[1]

for name, kwargs in configs:
    torch.manual_seed(0)
    with torch.no_grad():
        out = model.generate(**fact_inputs, max_new_tokens=15,
                              pad_token_id=tokenizer.pad_token_id, **kwargs)
    text = tokenizer.decode(out[0, fact_prompt_len:], skip_special_tokens=True)
    print(f"[{name}] {text!r}")

print(
    "\nTen sam zestaw strategii dekodowania, ale na pytaniu faktograficznym - "
    "rozklad jest dużo bardziej skoncentrowany (model jest 'pewny' odpowiedzi), "
    "wiec RÓŻNORODNOSC miedzy strategiami jest tu znacznie mniejsza niz na "
    "otwartym prompcie wyzej. Wieksza roznorodnosc (wyzej) NIE oznacza automatycznie "
    "lepszej jakosci - to wlasciwosc rozkladu dla danego promptu, nie uniwersalna "
    "zaleta sampling."
)


=== skoncentrowany, faktograficzny prompt (dla porownania) ===
[greedy] 'The capital of France is Paris.'


[sampling, tau=0.7] 'The capital of France is Paris. It is a city known for its rich'
[sampling, tau=1.5] 'The capital of France, and known as the City of Learn and Explore,'


[top-k=10] 'The capital of France is Paris. It is located in the northern part of'
[top-p=0.9] 'The capital of France is Paris. It is located in the northern part of'

Ten sam zestaw strategii dekodowania, ale na pytaniu faktograficznym - rozklad jest dużo bardziej skoncentrowany (model jest 'pewny' odpowiedzi), wiec RÓŻNORODNOSC miedzy strategiami jest tu znacznie mniejsza niz na otwartym prompcie wyzej. Wieksza roznorodnosc (wyzej) NIE oznacza automatycznie lepszej jakosci - to wlasciwosc rozkladu dla danego promptu, nie uniwersalna zaleta sampling.


In [15]:
# powtarzalnosc samplingu z ustalonym ziarnem
print("=== powtarzalnosc z ustalonym ziarnem ===")
for trial in range(2):
    torch.manual_seed(42)
    with torch.no_grad():
        out = model.generate(**dec_inputs, max_new_tokens=15, do_sample=True, temperature=0.8,
                              top_k=50, pad_token_id=tokenizer.pad_token_id)
    text = tokenizer.decode(out[0, dec_prompt_len:], skip_special_tokens=True)
    print(f"  trial {trial} (seed=42): {text!r}")
print("Identyczne na obu przebiegach - ustalone ziarno daje powtarzalnosc, mimo samplingu.")


=== powtarzalnosc z ustalonym ziarnem ===
  trial 0 (seed=42): 'One day, in a world where machines can play the piano, a robot'


  trial 1 (seed=42): 'One day, in a world where machines can play the piano, a robot'
Identyczne na obu przebiegach - ustalone ziarno daje powtarzalnosc, mimo samplingu.


## D. Kontekst i chat templates


In [16]:
plain_prompt = "What is the capital of France?"
plain_ids = tokenizer(plain_prompt, return_tensors="pt")
print(f"zwykly tekst (bez chat template): {plain_ids['input_ids'].shape[1]} tokenow")
print(tokenizer.decode(plain_ids["input_ids"][0]))

chat_messages = [{"role": "user", "content": plain_prompt}]
chat_text = tokenizer.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)
print(f"\nz chat template:\n{chat_text!r}")
# return_dict=True: apply_chat_template z tokenize=True zwraca BatchEncoding (slownik
# input_ids/attention_mask), NIE goly tensor - jawne return_dict=True czyni to widoczne
# w kodzie, zamiast zakladac ksztalt wyniku
chat_inputs = tokenizer.apply_chat_template(
    chat_messages, tokenize=True, add_generation_prompt=True,
    return_tensors="pt", return_dict=True,
)
print(f"\nz chat template: {chat_inputs['input_ids'].shape[1]} tokenow "
      f"(system prompt dodany automatycznie)")


zwykly tekst (bez chat template): 7 tokenow
What is the capital of France?

z chat template:
'<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nWhat is the capital of France?<|im_end|>\n<|im_start|>assistant\n'

z chat template: 37 tokenow (system prompt dodany automatycznie)


In [17]:
with torch.no_grad():
    out_plain = model.generate(**plain_ids, max_new_tokens=20, do_sample=False,
                                pad_token_id=tokenizer.pad_token_id)
    out_chat = model.generate(**chat_inputs, max_new_tokens=20, do_sample=False,
                               pad_token_id=tokenizer.pad_token_id)

chat_prompt_len = chat_inputs["input_ids"].shape[1]
plain_prompt_len = plain_ids["input_ids"].shape[1]
text_plain = tokenizer.decode(out_plain[0, plain_prompt_len:], skip_special_tokens=True)
text_chat = tokenizer.decode(out_chat[0, chat_prompt_len:], skip_special_tokens=True)
print(f"bez chat template: {text_plain!r}")
print(f"z chat template:   {text_chat!r}")
print(
    "\nModel instruct byl trenowany na formacie chat template - podanie 'zwyklego' "
    "tekstu nie jest bledem skladniowym, ale to wejscie z rozkladu, ktorego model "
    "'formalnie' nie widzial podczas fine-tuningu na rozmowy."
)


bez chat template: ''
z chat template:   'The capital of France is Paris.'

Model instruct byl trenowany na formacie chat template - podanie 'zwyklego' tekstu nie jest bledem skladniowym, ale to wejscie z rozkladu, ktorego model 'formalnie' nie widzial podczas fine-tuningu na rozmowy.


In [18]:
# koszt tokenowy historii rozmowy - rosnie z liczba tur, NIE jest 'pamiecia za darmo'
history = []
turns = [
    "My name is Kasia.",
    "What is my name?",
]
print("=== koszt tokenowy rosnacej historii rozmowy ===")
for turn in turns:
    history.append({"role": "user", "content": turn})
    hist_inputs = tokenizer.apply_chat_template(history, tokenize=True, add_generation_prompt=True,
                                                 return_tensors="pt", return_dict=True)
    hist_len = hist_inputs["input_ids"].shape[1]
    print(f"po dodaniu '{turn}': historia = {hist_len} tokenow")
    with torch.no_grad():
        out = model.generate(**hist_inputs, max_new_tokens=20, do_sample=False,
                              pad_token_id=tokenizer.pad_token_id)
    reply = tokenizer.decode(out[0, hist_len:], skip_special_tokens=True)
    print(f"  odpowiedz: {reply!r}")
    history.append({"role": "assistant", "content": reply})

print(
    "\nDruga tura 'pamieta' imie TYLKO dlatego, ze CALA pierwsza tura (user+assistant) "
    "zostala jawnie dolaczona do wejscia drugiego wywolania - model sam z siebie nie "
    "'pamieta' niczego miedzy wywolaniami generate()."
)


=== koszt tokenowy rosnacej historii rozmowy ===
po dodaniu 'My name is Kasia.': historia = 36 tokenow


  odpowiedz: "Kasia, I'm glad you're here to help with your creative writing endeavors. I'm here"
po dodaniu 'What is my name?': historia = 72 tokenow


  odpowiedz: "My name is Kasia, and I'm a creative writing assistant. I'm here to help you"

Druga tura 'pamieta' imie TYLKO dlatego, ze CALA pierwsza tura (user+assistant) zostala jawnie dolaczona do wejscia drugiego wywolania - model sam z siebie nie 'pamieta' niczego miedzy wywolaniami generate().


## E. KV cache i koszt


In [19]:
long_prompt = [{"role": "user", "content":
    "Describe in detail the water cycle, including evaporation, condensation, and precipitation."}]
cache_inputs = tokenizer.apply_chat_template(
    long_prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True
)
cache_prompt_len = cache_inputs["input_ids"].shape[1]
print(f"prompt: {cache_prompt_len} tokenow")

print(f"\n{'n_new':>6} {'use_cache':>10} {'czas (s)':>9} {'tok/s':>8}")
for n_new in [16, 32, 64, 128]:
    for use_cache in [True, False]:
        torch.manual_seed(0)
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(
                **cache_inputs, min_new_tokens=n_new, max_new_tokens=n_new, do_sample=False,
                use_cache=use_cache, pad_token_id=tokenizer.pad_token_id,
            )
        dt = time.time() - t0
        n_generated = out.shape[1] - cache_prompt_len
        print(f"{n_new:>6} {str(use_cache):>10} {dt:>9.3f} {n_generated/dt:>8.1f}")

print(
    "\nuse_cache=True jest wyraznie szybsze, z PRZEWAGA ROSNACA wraz z dlugoscia "
    "generowania - bez cache, kazdy nowy krok przelicza attention dla WSZYSTKICH "
    "dotychczasowych pozycji od zera (koszt kroku rosnie z pozycja), z cache - "
    "kazdy krok korzysta z zapisanych K/V (koszt kroku ~staly)."
)


prompt: 45 tokenow

 n_new  use_cache  czas (s)    tok/s
    16       True     0.177     90.2


    16      False     0.442     36.2


    32       True     0.330     96.9


    32      False     0.999     32.0


    64       True     0.619    103.4


    64      False     2.314     27.7


   128       True     1.293     99.0


   128      False     5.642     22.7

use_cache=True jest wyraznie szybsze, z PRZEWAGA ROSNACA wraz z dlugoscia generowania - bez cache, kazdy nowy krok przelicza attention dla WSZYSTKICH dotychczasowych pozycji od zera (koszt kroku rosnie z pozycja), z cache - kazdy krok korzysta z zapisanych K/V (koszt kroku ~staly).


In [20]:
# wplyw dlugosci PROMPTU (fixed generowanie=32 tokeny)
short_p = [{"role": "user", "content": "Hello"}]
filler = "The quick brown fox jumps over the lazy dog. " * 40
long_text = "Please summarize in one sentence: " + filler
long_p = [{"role": "user", "content": long_text}]

print(f"{'prompt':40} {'prompt_len':>10} {'czas (s)':>9}")
for label, msgs in [("short", short_p), ("long", long_p)]:
    ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                         return_tensors="pt", return_dict=True)
    plen = ids["input_ids"].shape[1]
    torch.manual_seed(0)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**ids, min_new_tokens=32, max_new_tokens=32, do_sample=False,
                              pad_token_id=tokenizer.pad_token_id)
    dt = time.time() - t0
    print(f"{label:40} {plen:>10} {dt:>9.3f}")

print(
    "\nDluzszy prompt kosztuje wiecej w PREFILL (jeden przebieg nad calym promptem), "
    "ale dla krotkiego generowania (32 tokeny) DECODE wciaz dominuje calkowity czas - "
    "efekt dlugosci promptu jest realny, ale nie dominujacy przy tej skali."
)


prompt                                   prompt_len  czas (s)


short                                            31     0.317


long                                            437     0.469

Dluzszy prompt kosztuje wiecej w PREFILL (jeden przebieg nad calym promptem), ale dla krotkiego generowania (32 tokeny) DECODE wciaz dominuje calkowity czas - efekt dlugosci promptu jest realny, ale nie dominujacy przy tej skali.


In [21]:
# KV cache - koszt pamieci: wzor vs rzeczywisty rozmiar tensorow
sample_inputs = tokenizer("Hello, how are you today? I am testing the key-value cache.",
                           return_tensors="pt")
with torch.no_grad():
    out = model(**sample_inputs, use_cache=True)
cache = out.past_key_values
seq_len = sample_inputs["input_ids"].shape[1]

n_layers = model.config.num_hidden_layers
n_kv_heads = model.config.num_key_value_heads
head_dim = model.config.head_dim
bytes_per_elem = 4  # float32

formula_bytes = 2 * n_layers * n_kv_heads * head_dim * seq_len * bytes_per_elem


def layer_bytes(layer):
    return (layer.keys.numel() * layer.keys.element_size()
            + layer.values.numel() * layer.values.element_size())


actual_bytes = sum(layer_bytes(layer) for layer in cache.layers)

print(f"model: {n_layers} warstw, {model.config.num_attention_heads} glow Q, "
      f"{n_kv_heads} glow KV (GQA!), head_dim={head_dim}")
print(f"sekwencja: {seq_len} tokenow")
print(f"wzor:      2 * {n_layers} * {n_kv_heads} * {head_dim} * {seq_len} * {bytes_per_elem} "
      f"= {formula_bytes} bajtow = {formula_bytes/1024:.1f} KB")
print(f"rzeczywisty rozmiar tensorow cache: {actual_bytes} bajtow = {actual_bytes/1024:.1f} KB")
print(f"zgodne (bit-exact)?: {formula_bytes == actual_bytes}")
assert formula_bytes == actual_bytes

print(
    f"\nTen model ma {model.config.num_attention_heads} glow zapytan (Q), ale TYLKO "
    f"{n_kv_heads} glowy KV (grouped-query attention) - wzor MUSI uzywac liczby glow KV, "
    "nie liczby glow Q, inaczej koszt pamieci bylby przeszacowany 3x."
)


model: 30 warstw, 9 glow Q, 3 glow KV (GQA!), head_dim=64
sekwencja: 16 tokenow
wzor:      2 * 30 * 3 * 64 * 16 * 4 = 737280 bajtow = 720.0 KB
rzeczywisty rozmiar tensorow cache: 737280 bajtow = 720.0 KB
zgodne (bit-exact)?: True

Ten model ma 9 glow zapytan (Q), ale TYLKO 3 glowy KV (grouped-query attention) - wzor MUSI uzywac liczby glow KV, nie liczby glow Q, inaczej koszt pamieci bylby przeszacowany 3x.


## Podsumowanie

- Tokenizacja: ten sam tokenizer daje ~1,1 tokenów/słowo dla angielskiego i
  ~3,6 dla polskiego — zmierzone, nie przybliżone. Liczby dzielone na
  pojedyncze cyfry; rzadkie słowa rozbite na podsłowa, nie `<UNK>`.
- Ręczna pętla generowania (model + argmax + konkatenacja) daje **identyczne**
  tokeny co `model.generate(do_sample=False)`.
- Temperature zmienia **ostrość** rozkładu, nie wybiera tokenu — różnorodność
  między strategiami dekodowania jest dużo mniejsza na skoncentrowanym
  (faktograficznym) prompcie niż na otwartym (kreatywnym) — różnorodność nie
  jest uniwersalną właściwością samplingu, zależy od rozkładu dla danego
  promptu.
- Historia rozmowy musi być jawnie dołączona do każdego wywołania — model
  nie ma pamięci między wywołaniami `generate()`.
- `use_cache=True` jest wyraźnie szybsze niż `False`, z przewagą **rosnącą**
  z długością generowania. Koszt pamięci KV cache zgadza się bit-exact ze
  wzorem $2NH_{kv}d_{\text{head}}T \cdot \text{bytes}$ — model ma $9$ głów
  zapytań, ale tylko $3$ głowy KV (GQA), co zmniejsza koszt pamięci cache
  $3\times$ względem naiwnego liczenia po głowach zapytań.

Pytania do dyskusji i bibliografia: zobacz [`lecture.md`](lecture.md).
